# GitHub Metrics: Dimensionality Reduction (Mid-Report 2, Section 2)

**Scope:** dedicated deep-dive on the dimensionality-reduction piece of the mid-report. Runs on
`data/merged_dataset.csv` with the **same population and feature matrix** as the clustering section
(`notebooks/github_clustering.ipynb`) and the outlier detection section (`notebooks/outlier_detection.ipynb`):
the 7,890 real projects (active in at least 3 months of 2024) described by the 14 standardized 2024 activity
features. Using one shared matrix means every number in this section lines up directly with the other two,
which is what the Integration and Insights section needs.

**House rules:** reads only `merged_dataset.csv`; figures to `figures/dr_*.svg`; deterministic seeds
everywhere so every number is reproducible.


## Settings (for the report's Integration and Insights section)

This section uses exactly the same setup as the other two sections, so no swapping is needed:

| Item | Value |
|---|---|
| Population | **7,890 real projects**: `active_months_2024 >= 3` (of 62,627 repos), identical to `github_clustering.ipynb` and `outlier_detection.ipynb` |
| Features | The same 14 as both sections: 8 volume counts (`push_events_2024`, `issue_events_2024`, `issue_comment_events_2024`, `pull_request_events_2024`, `review_events_2024`, `release_events_2024`, `stars_received_2024`, `fork_events_2024`), 3 breadth (`distinct_actors_2024`, `active_days_2024`, `active_months_2024`), 2 trend (`qualifying_events_h1_2024`, `qualifying_events_h2_2024`), 1 recency (`days_since_last_push_at_cutoff`) |
| Preprocessing | `days_since_last_push_at_cutoff` NaN (no push in 2024) encoded as 366, then log1p on **all 14 features**, then StandardScaler. Identical to `github_clustering.ipynb`. Note: `outlier_detection.ipynb`'s own `build_features` leaves recency unlogged, so its K-means/PCA sizes differ by a few projects (documented in its Section 9) |
| DR: PCA | Full matrix; components kept for the variance story and for the clustering-comparison test (Section 3) |
| DR: t-SNE | n_components = 2, perplexity = 30 (sweep 5/30), init = "pca", learning_rate = "auto", max_iter = 1000, random_state = 42; deterministic 3,000-repo subsample (seed 42) |
| DR: UMAP | Not applied (not required by the rubric; noted as an optional extension) |
| Clustering used for comparison | K-means **K = 2** (the setting selected in `github_clustering.ipynb`; n_init = 20, seed 42), compared on the full 14-d matrix vs the first 3 PCA components |

**Setup alignment:** population and features are shared with both sections; the transform is byte-for-byte
the `github_clustering.ipynb` matrix, so silhouette scores, cluster assignments (6,895 / 995), and the PCA
map line up directly with the clustering section. Against the outlier section's own matrix only recency is
transformed differently, which shifts a handful of boundary projects (documented in that notebook).


## 1. The 'before' view: why 14 correlated features need reducing

The feature matrix has 14 numeric columns, several of which measure the same underlying thing (event volume,
breadth, half-year trend). A correlation heatmap shows the redundancy; no single 2D slice of the raw space
captures the structure, which is exactly what DR is for.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA, FIG = ROOT / "data", ROOT / "figures"

# --- identical setup to github_clustering.ipynb and outlier_detection.ipynb ---
df_all = pd.read_csv(DATA / "merged_dataset.csv")
df = df_all.loc[df_all["active_months_2024"] >= 3].copy()

VOLUME = ["push_events_2024", "issue_events_2024", "issue_comment_events_2024",
          "pull_request_events_2024", "review_events_2024", "release_events_2024",
          "stars_received_2024", "fork_events_2024"]
BREADTH = ["distinct_actors_2024", "active_days_2024", "active_months_2024"]
TREND = ["qualifying_events_h1_2024", "qualifying_events_h2_2024"]
RECENCY = ["days_since_last_push_at_cutoff"]
FEATURES = VOLUME + BREADTH + TREND + RECENCY

raw = df[FEATURES].copy()
raw[RECENCY] = raw[RECENCY].fillna(366)
# identical transform to github_clustering.ipynb: log1p on ALL 14 features (recency included)
X = StandardScaler().fit_transform(np.log1p(raw))
X = pd.DataFrame(X, index=df.index, columns=FEATURES).values

print(f"real projects: {len(df):,} of {len(df_all):,}")
print("feature matrix:", X.shape)

# correlation heatmap of the raw (untransformed) features
corr = df[FEATURES].corr()
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES))); ax.set_yticks(range(len(FEATURES)))
short = [f.replace("_events_2024", "").replace("_received_2024", "")
         .replace("_at_cutoff", "").replace("qualifying_events_h1_2024", "H1 events")
         .replace("qualifying_events_h2_2024", "H2 events") for f in FEATURES]
ax.set_xticklabels(short, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(short, fontsize=7)
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(corr.values[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation of the 14 raw features (real projects)")
plt.tight_layout()
fig.savefig(FIG / "dr_correlation_heatmap.svg")
plt.close(fig)

iup = np.triu_indices(len(FEATURES), k=1)
strong = [(FEATURES[a], FEATURES[b], round(float(corr.values[a, b]), 2))
          for a, b in zip(*iup) if abs(corr.values[a, b]) >= 0.6]
print(f"feature pairs with |corr| >= 0.6: {len(strong)} of {len(iup[0])}")
for a, b, r in strong:
    print(f"  {a} x {b}: {r:+.2f}")
print("saved dr_correlation_heatmap.svg")

real projects: 7,890 of 62,627
feature matrix: (7890, 14)


feature pairs with |corr| >= 0.6: 11 of 91
  push_events_2024 x qualifying_events_h1_2024: +0.70
  push_events_2024 x qualifying_events_h2_2024: +0.99
  issue_events_2024 x issue_comment_events_2024: +0.96
  issue_events_2024 x distinct_actors_2024: +0.65
  issue_events_2024 x qualifying_events_h1_2024: +0.61
  issue_comment_events_2024 x distinct_actors_2024: +0.76
  issue_comment_events_2024 x qualifying_events_h1_2024: +0.65
  pull_request_events_2024 x review_events_2024: +0.76
  stars_received_2024 x fork_events_2024: +0.73
  active_days_2024 x active_months_2024: +0.63
  qualifying_events_h1_2024 x qualifying_events_h2_2024: +0.72
saved dr_correlation_heatmap.svg


### Notes for the report:
- The 14 raw features are redundant: **11 of the 91 feature pairs correlate at |r| >= 0.6**. The strongest are push events x H2 qualifying events (+0.99), issues x issue comments (+0.96), and three mid-strength blocks: PRs x reviews (+0.76), issue comments x distinct actors (+0.76), and stars x forks (+0.73). Volume, breadth, and the half-year trend features largely measure the same underlying activity level.
- This is the one-paragraph justification for reducing before clustering: most of the 14 dimensions are near-duplicates of a few latent axes.


## 2. PCA: linear and variance-maximizing

PCA gives the global picture: which directions carry the most variance, and how much of the total is
captured by a small number of components. Fitted on all 7,890 real projects; the scree plot and loadings
make the axes *interpretable*, not just numeric.


In [2]:
from sklearn.decomposition import PCA

pca = PCA(n_components=X.shape[1], random_state=42)
pca.fit(X)
evr = pca.explained_variance_ratio_
print("variance explained per component:")
for i in range(len(evr)):
    print(f"  PC{i+1}: {evr[i]*100:5.1f}%   cumulative {np.cumsum(evr)[i]*100:5.1f}%")
k80 = int(np.searchsorted(np.cumsum(evr), 0.80) + 1)
print(f"components needed for >= 80% variance: {k80}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(evr)+1), evr*100, color="#4c8bf5", alpha=0.9)
axes[0].axvline(k80, color="black", ls="--", lw=1)
axes[0].text(k80 + 0.1, 2, f"k={k80} for 80%", fontsize=9)
axes[0].set_title("PCA scree plot")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance %")
ax_c = axes[0].twinx()
ax_c.plot(range(1, len(evr)+1), np.cumsum(evr)*100, "o-", color="#d63384", ms=4)
ax_c.set_ylabel("cumulative %"); ax_c.set_ylim(0, 105)

load = pd.DataFrame(pca.components_[:2].T, index=FEATURES, columns=["PC1", "PC2"]).round(3)
ax = axes[1]
y = np.arange(len(FEATURES))
ax.barh(y - 0.2, load["PC1"], height=0.38, color="#4c8bf5", label="PC1")
ax.barh(y + 0.2, load["PC2"], height=0.38, color="#e07b39", label="PC2")
ax.set_yticks(y); ax.set_yticklabels(short, fontsize=7)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Loadings: PC1 vs PC2")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_pca_scree_and_loadings.svg")
plt.close(fig)

print(load.to_string())
print("saved dr_pca_scree_and_loadings.svg")

variance explained per component:
  PC1:  40.8%   cumulative  40.8%
  PC2:  18.2%   cumulative  59.1%
  PC3:   9.6%   cumulative  68.6%
  PC4:   7.1%   cumulative  75.7%
  PC5:   5.9%   cumulative  81.6%
  PC6:   4.0%   cumulative  85.6%
  PC7:   3.4%   cumulative  89.0%
  PC8:   2.9%   cumulative  91.9%
  PC9:   2.0%   cumulative  93.9%
  PC10:   1.7%   cumulative  95.7%
  PC11:   1.5%   cumulative  97.2%
  PC12:   1.1%   cumulative  98.3%
  PC13:   1.1%   cumulative  99.4%
  PC14:   0.6%   cumulative 100.0%
components needed for >= 80% variance: 5


                                  PC1    PC2
push_events_2024                0.191  0.441
issue_events_2024               0.301 -0.219
issue_comment_events_2024       0.334 -0.212
pull_request_events_2024        0.274 -0.106
review_events_2024              0.280 -0.116
release_events_2024             0.188 -0.066
stars_received_2024             0.279 -0.253
fork_events_2024                0.292 -0.265
distinct_actors_2024            0.346 -0.231
active_days_2024                0.283  0.383
active_months_2024              0.284  0.289
qualifying_events_h1_2024       0.225  0.144
qualifying_events_h2_2024       0.240  0.320
days_since_last_push_at_cutoff -0.143 -0.379
saved dr_pca_scree_and_loadings.svg


In [3]:
# fit the same K-means as github_clustering.ipynb (K=2, n_init=20, seed 42) for coloring
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, n_init=20, random_state=42).fit(X)
print("k-means cluster sizes:", pd.Series(kmeans.labels_).value_counts().sort_index().to_dict())

# 2D scatter on PC1/PC2, colored by the K=2 k-means assignment (the 'after' view)
Xp = pca.transform(X)
fig, ax = plt.subplots(figsize=(9, 7))
for j in range(2):
    m = kmeans.labels_ == j
    ax.scatter(Xp[m, 0], Xp[m, 1], s=6, alpha=0.35,
               color="#4c8bf5" if j == 0 else "#e07b39", label=f"k-means cluster {j}")
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
ax.set_title("PCA 2D projection, colored by K=2 k-means cluster")
ax.legend(edgecolor="none", markerscale=3)
plt.tight_layout()
fig.savefig(FIG / "dr_pca_scatter_clusters.svg")
plt.close(fig)
print("saved dr_pca_scatter_clusters.svg")

k-means cluster sizes: {0: 6895, 1: 995}


saved dr_pca_scatter_clusters.svg


**Reading the loadings.** PC1 (40.8% of variance) loads positively on all 13 activity features and negatively
on days-since-last-push (-0.143): it is an overall **activity/size axis** (distinct actors +0.346, issue comments
+0.334, issues +0.301), with more active projects also being the ones that pushed most recently. PC2 (18.2%)
contrasts **push cadence** (push events +0.441, active days +0.383, H2 events +0.320) against **discussion and
adoption engagement** (forks -0.265, stars -0.253, distinct actors -0.231, issues -0.219): one axis separates
commit-heavy projects from issue/star-heavy ones. Both axes are interpretable in project terms, which PCA's
variance-maximizing construction does not guarantee.

### Notes for the report:
- Variance: PC1 40.8%, PC2 18.2% (59.1% cumulative), PC3 9.6% (68.6%), PC4 7.1% (75.7%), PC5 5.9% (**81.6%**). Five components reach the 80% line; the scree plot shows no elbow before that.
- Both axes are interpretable in project terms: PC1 = overall activity/size, PC2 = push cadence vs discussion/adoption engagement (loadings table above). Cite the loadings table, not just the scree plot.
- PCA is the workhorse for downstream modelling: its axes have meaning (unlike t-SNE), so it is the right choice for the clustering comparison in Section 3.


## 3. Does DR help clustering? (before vs after)

The rubric asks whether dimensionality reduction improved separation or interpretability. Test: k-means
(K = 2, the setting selected in `github_clustering.ipynb`; n_init = 20, seed 42) on (a) the full
standardized 14-feature matrix versus (b) the first three PCA components. Silhouette uses the same
sampled protocol as the clustering section (sample_size = 2000, seed 42).


In [4]:
from sklearn.metrics import silhouette_score, adjusted_rand_score

km_full = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(X)
sil_full = silhouette_score(X, km_full, sample_size=min(2000, len(X)), random_state=42)

Xp3 = Xp[:, :3]
km_pca = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(Xp3)
sil_pca = silhouette_score(Xp3, km_pca, sample_size=min(2000, len(Xp3)), random_state=42)

print(f"k-means on full 14-d matrix:  silhouette = {sil_full:.3f}")
print(f"k-means on first 3 PCA components: silhouette = {sil_pca:.3f}")
print(f"ARI (label agreement between the two):     {adjusted_rand_score(km_full, km_pca):.3f}")

# cross-check against github_clustering.ipynb: inactivity rate per cluster (validation only)
df["kmeans"] = km_full
print("\nfuture inactivity by k-means cluster (should match the clustering section):")
print(df.groupby("kmeans")["inactive_next_365d"].agg(repositories="size", inactivity_rate="mean").round(3).to_string())

k-means on full 14-d matrix:  silhouette = 0.475
k-means on first 3 PCA components: silhouette = 0.535
ARI (label agreement between the two):     0.987

future inactivity by k-means cluster (should match the clustering section):
        repositories  inactivity_rate
kmeans                               
0               6895            0.525
1                995            0.124


**Result.** K-means on the full standardized 14-feature matrix scores silhouette **0.475** (identical to the
clustering section's reported value, confirming the shared matrix). On only the first three PCA components the
score rises to **0.535** (+0.060) while label agreement with the full-matrix fit stays at ARI **0.987**. The two
fits recover essentially the same partition (6,895 vs 995 repositories; future-inactivity rates 52.5% vs 12.4%,
matching the clustering section exactly), so keeping three components discards redundant noise rather than
structure: dimensionality reduction *improved* separation here without changing what was found.


## 4. t-SNE: nonlinear and local-structure preserving

t-SNE preserves *local* neighborhoods rather than global variance, giving a complementary view of the
same data. It is stochastic (seeded here) and its distances are not meaningful beyond the plot.
A perplexity sweep justifies the parameter choice: too low fragments the data, too high blurs local
structure. We run on a deterministic 3,000-repo subsample for tractability.


In [5]:
from sklearn.manifold import TSNE

rng = np.random.default_rng(42)
sub_idx = rng.choice(len(X), size=3000, replace=False); sub_idx.sort()
Xs, dfs = X[sub_idx], df.iloc[sub_idx].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, perp in zip(axes, [5, 30]):
    ts = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca",
              learning_rate="auto", max_iter=1000)
    Xt = ts.fit_transform(Xs)
    for j in range(2):
        m = dfs["kmeans"] == j
        ax.scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35,
                   color="#4c8bf5" if j == 0 else "#e07b39", label=f"k-means cluster {j}")
    ax.set_title(f"t-SNE perplexity={perp}")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(edgecolor="none", markerscale=3, fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_tsne_perplexity_sweep.svg")
plt.close(fig)
print("saved dr_tsne_perplexity_sweep.svg")

saved dr_tsne_perplexity_sweep.svg


In [6]:
# final t-SNE plot (perplexity 30): k-means cluster and the 2025 outcome (validation only)
ts = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto", max_iter=1000)
Xt = ts.fit_transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for j in range(2):
    m = dfs["kmeans"] == j
    axes[0].scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35,
                    color="#4c8bf5" if j == 0 else "#e07b39", label=f"k-means cluster {j}")
axes[0].set_title("t-SNE (perp=30) by k-means cluster")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].legend(edgecolor="none", markerscale=3)

for v, c in [(0, "#2e7d68"), (1, "#c85a54")]:
    m = dfs["inactive_next_365d"] == v
    axes[1].scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35,
                    color=c, label=f"inactive in 2025: {v}")
axes[1].set_title("t-SNE (perp=30) by 2025 outcome (validation only)")
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].legend(edgecolor="none", markerscale=3, fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_tsne_final.svg")
plt.close(fig)
print("saved dr_tsne_final.svg")

saved dr_tsne_final.svg


**PCA vs t-SNE: what each is best at.** PCA: global, interpretable axes, valid distances, variance
explained. Use it for modelling and to satisfy the 'proportion of variance explained' requirement.
t-SNE: local neighborhoods, high-contrast separation (perplexity 30), no meaningful distances. Use it
for figures only, per Lesson 10.4.3's practical advice.

### Notes for the report:
- Perplexity 30 selected from a 5-vs-30 sweep (Lesson 10.4.3: experiment with perplexity); deterministic 3,000-repo subsample (scalability caveat, Lesson 10.4.1).
- t-SNE is visualization-only: axes have no meaning; never feed the embedding into models.


## 5. PCA vs t-SNE: comparison table

| Technique | Parameters | What it preserves | Key result | Figure |
|---|---|---|---|---|
| PCA | full matrix, 14 components fitted | global variance | PC1 40.8%, PC2 18.2%; 5 components for >= 80% (81.6%) | dr_pca_scree_and_loadings.svg |
| PCA scatter | first 2 components | interpretable axes | low-activity group compact on the left, high-activity group fanning out to its right with an overlap band between | dr_pca_scatter_clusters.svg |
| t-SNE | perplexity 30, deterministic 3,000-repo subsample | local neighborhood | distinct cluster blobs; perp=5 fragments | dr_tsne_final.svg, dr_tsne_perplexity_sweep.svg |
| DR -> clustering | k-means K=2 on full 14-d vs first 3 PCA components | cluster stability | silhouette 0.475 -> 0.535; ARI 0.987 | (cell output, Section 3) |


## 6. Summary for the report

### TL;DR

- **What was done:** PCA (linear) + t-SNE (nonlinear) on the shared real-project matrix (7,890 repos, 14 standardized 2024 activity features, identical to the clustering section).
- **Variance explained:** PC1 40.8%, PC2 18.2% (59.1% cumulative); five components reach 81.6%. PC1 is an overall activity/size axis; PC2 contrasts push cadence with discussion/adoption engagement.
- **Impact on clustering:** k-means K=2 silhouette improves from 0.475 (full 14-d) to 0.535 (first 3 PCA components) with ARI 0.987 label agreement: reduction helps and changes nothing substantive.
- **t-SNE** (perplexity 30, deterministic 3,000-repo subsample) shows the small high-activity cluster as a distinct dense blob while the large low-activity group spreads diffusely across the rest of the embedding; visualization-only per Lesson 10.4.3.
- **Not applied:** Kernel PCA (n x n kernel-matrix cost, no evidence of a dominant nonlinear structure in the scree); UMAP (optional final-report extension).

### Results summary

| Question | Answer |
|---|---|
| Why reduce? | 14 features, 11 of 91 pairs at |r| >= 0.6; volume/breadth/trend are near-duplicates (Section 1) |
| How much variance survives? | PC1 40.8%, PC2 18.2%; 5 components for >= 80% (81.6%) |
| Are the axes interpretable? | Yes: PC1 = activity/size, PC2 = push cadence vs discussion/adoption engagement |
| Does DR help clustering? | Silhouette 0.475 -> 0.535 with ARI 0.987 (Section 3) |
| Local structure check | t-SNE perplexity 30: the high-activity cluster forms a distinct dense blob; the low-activity group spreads diffusely |

### Report write-up (copy into the report)

One subsection per figure: file name, caption, insight.

**Figure 1: `dr_correlation_heatmap.svg`** (SVG in `figures/`). Caption: *Correlation of the 14 raw features (7,890 real projects).* Insight: redundancy is concentrated among volume counts and between volume and breadth (push x H2 events +0.99; issues x issue comments +0.96). This motivates reduction.

**Figure 2: `dr_pca_scree_and_loadings.svg`** (core figure if space is short). Caption: *PCA scree plot with cumulative variance and PC1/PC2 loadings.* Insight: five components carry 81.6% of the variance, and both leading axes are interpretable (activity/size; push cadence vs engagement).

**Figure 3: `dr_pca_scatter_clusters.svg`** (core figure if space is short). Caption: *PCA 2D projection, colored by K=2 k-means cluster.* Insight: the two activity profiles from the clustering section (6,895 vs 995 repositories) mostly separate along PC1 (the activity axis): the large low-activity group forms a compact wedge on the left and the small high-activity group fans out to its right, with an overlap band between them.

**Figure 4: `dr_tsne_perplexity_sweep.svg`**. Caption: *t-SNE perplexity sweep (5 vs 30) on the 3,000-repo subsample.* Insight: perplexity 5 breaks the data into many small blobs; 30 preserves larger groupings while keeping local detail, so it is selected.

**Figure 5: `dr_tsne_final.svg`** (core figure if space is short). Caption: *t-SNE (perplexity 30) by k-means cluster and by 2025 outcome.* Insight: the high-activity cluster is a cohesive blob while the low-activity group spreads diffusely, consistent with that group being heterogeneous; the 2025-outcome panel shows the association with decay without using it in any fit.
